# Select patients from stroke registry data
- Inclusion criteria: > 17y, ischemic stroke, inpatient/non-transferred, not refusing to participate
- Exclusion criteria: < 12h

In [ ]:
import pandas as pd
import numpy as np
import os

In [ ]:
stroke_registry_data_path = '/Users/jk1/OneDrive - unige.ch/stroke_research/geneva_stroke_unit_dataset/data/stroke_registry/post_hoc_modified/stroke_registry_post_hoc_modified.xlsx'

In [ ]:
manual_eds_completion_folder = '/Users/jk1/OneDrive - unige.ch/stroke_research/geneva_stroke_unit_dataset/data/stroke_registry/manuel_eds_completion'

In [ ]:
output_path = '/Users/jk1/temp/opsum_extration_output'

In [ ]:
all_data_df = pd.read_excel(stroke_registry_data_path)

In [ ]:
from preprocessing.utils import create_registry_case_identification_column

all_data_df['patient_id'] = all_data_df['Case ID'].apply(lambda x: x[8:-4]).astype(str)
all_data_df['EDS_last_4_digits'] = all_data_df['Case ID'].apply(lambda x: x[-4:]).astype(str)
all_data_df['case_admission_id'] = create_registry_case_identification_column(all_data_df)

In [ ]:
print('Number of records screened: ', len(all_data_df['case_admission_id'].unique()))

### Exclude patients refusing participation in research

In [ ]:
# Remove patients not wanting to participate in research
print(all_data_df['Patient refuses use of data for research'].value_counts())
full_data_df = all_data_df[all_data_df['Patient refuses use of data for research'] != 'yes']

### Include only ischemic stroke patients

In [ ]:
full_data_df['Type of event'].value_counts()

In [ ]:
# select only ischemic stroke patients
all_stroke_df = full_data_df[full_data_df['Type of event'] == 'Ischemic stroke']

In [ ]:
print('Number of patients excluded because not ischemic stroke: ', len(full_data_df['case_admission_id'].unique()) - len(all_stroke_df['case_admission_id'].unique()))

### Exclude patients not hospitalised in our center or discharged

In [ ]:
all_stroke_df['Initial hospitalization'].value_counts()

In [ ]:
# exclude patients that were immediately discharged or referred to other center
stroke_df = all_stroke_df[all_stroke_df['Initial hospitalization'] != 'Outpatient management']
stroke_df = stroke_df[stroke_df['Initial hospitalization'] != 'Referral to other Stroke Unit or Stroke Center']
stroke_df = stroke_df[stroke_df['Initial hospitalization'] != 'Referral to other hospital or care institution']

In [ ]:
print('Number of patients excluded because discharged or referred to other center: ', len(all_stroke_df['case_admission_id'].unique()) - len(stroke_df['case_admission_id'].unique()))

### Exclude patients with less than 12h of hospitalization

In [ ]:
# set end of reference period to stroke onset or arrival at hospital, whichever is later
# this takes into account potential in-hospital stroke events

datatime_format = '%d.%m.%Y %H:%M'
stroke_df['arrival_dt'] = pd.to_datetime(stroke_df['Arrival at hospital'],
                                                  format='%Y%m%d').dt.strftime('%d.%m.%Y') + ' ' + \
                                   pd.to_datetime(stroke_df['Arrival time'], format='%H:%M',
                                                  infer_datetime_format=True).dt.strftime('%H:%M')

stroke_df['stroke_dt'] = pd.to_datetime(stroke_df['Onset date'],
                                                 format='%Y%m%d').dt.strftime('%d.%m.%Y') + ' ' + \
                                    pd.to_datetime(stroke_df['Onset time'], format='%H:%M',
                                                   infer_datetime_format=True).dt.strftime('%H:%M')

stroke_df['delta_onset_arrival'] = (
        pd.to_datetime(stroke_df['stroke_dt'], format=datatime_format, errors='coerce')
        - pd.to_datetime(stroke_df['arrival_dt'], format=datatime_format, errors='coerce')
).dt.total_seconds()
stroke_df['registry_sampling_start_upper_bound_reference'] = stroke_df \
    .apply(lambda x: x['stroke_dt'] if x['delta_onset_arrival'] > 0 else x['arrival_dt'],
           axis=1)


In [ ]:
stroke_df['discharge_dt'] = pd.to_datetime(stroke_df['Discharge date'],
                                                  format='%Y%m%d').dt.strftime('%d.%m.%Y') + ' ' + \
                                   pd.to_datetime(stroke_df['Discharge time'], format='%H:%M',
                                                  infer_datetime_format=True).dt.strftime('%H:%M')

stroke_df['death_dt'] = pd.to_datetime(stroke_df['Death at hospital date'],
                                                  format='%Y%m%d').dt.strftime('%d.%m.%Y') + ' ' + \
                                   pd.to_datetime(stroke_df['Death at hospital time'], format='%H:%M',
                                                  infer_datetime_format=True).dt.strftime('%H:%M')

stroke_df['registry_sampling_end'] = stroke_df['discharge_dt'].fillna(stroke_df['death_dt'])


In [ ]:
stroke_df['registry_sample_range'] = pd.to_datetime(stroke_df['registry_sampling_end'], format=datatime_format) \
                                                - pd.to_datetime(stroke_df['registry_sampling_start_upper_bound_reference'], format=datatime_format)

In [ ]:
cid_with_hospitalization_duration_less_than_12h = stroke_df[stroke_df['registry_sample_range'] < pd.Timedelta('12h')]['case_admission_id'].unique()

In [ ]:
print('Number of patients excluded because hospitalization duration less than 12h: ', len(cid_with_hospitalization_duration_less_than_12h))

In [ ]:
# exclude patients with less than 12h of hospitalization
stroke_df = stroke_df[~stroke_df['case_admission_id'].isin(cid_with_hospitalization_duration_less_than_12h)]

In [ ]:
len(stroke_df['case_admission_id'].unique())

In [ ]:
# counting patients with outcome variables
sum(stroke_df['3M mRS'].value_counts())

In [ ]:
stroke_df['Death in hospital'].value_counts()


In [ ]:
stroke_df['Referral'].value_counts()

In [ ]:
onset_date = pd.to_datetime(pd.to_datetime(stroke_df['Onset date'], format='%Y%m%d').dt.strftime('%d-%m-%Y') \
                                        + ' ' + stroke_df['Onset time'])

admission_date = pd.to_datetime(pd.to_datetime(stroke_df['Arrival at hospital'], format='%Y%m%d').dt.strftime('%d-%m-%Y') \
                                        + ' ' + stroke_df['Arrival time'])

discharge_date = pd.to_datetime(pd.to_datetime(stroke_df['Discharge date'], format='%Y%m%d').dt.strftime('%d-%m-%Y') \
                                        + ' ' + stroke_df['Discharge time'])


In [ ]:
stroke_df.head()

#### Fuse with databases of manually completed EDS

In [ ]:
manual_eds_completion_dfs = [pd.read_excel(os.path.join(manual_eds_completion_folder, f)) for f in os.listdir(manual_eds_completion_folder) if f.endswith('.xlsx')]

In [ ]:
all_manual_eds_completions = pd.concat(manual_eds_completion_dfs)

In [ ]:
all_manual_eds_completions = all_manual_eds_completions[['patient_id', 'EDS_last_4_digits', 'manual_eds', 'manual_patient_id']]
all_manual_eds_completions = all_manual_eds_completions.astype(str)
all_manual_eds_completions['manual_eds'] = all_manual_eds_completions['manual_eds'].str.rstrip('.0')
all_manual_eds_completions['manual_patient_id'] = all_manual_eds_completions['manual_patient_id'].str.rstrip('.0')

In [ ]:
all_manual_eds_completions

In [ ]:
stroke_df = stroke_df.merge(all_manual_eds_completions, how='left', on=['patient_id', 'EDS_last_4_digits'])

In [ ]:
stroke_df.head()

In [ ]:
high_frequency_data_patient_selection_with_details = stroke_df[['patient_id', 'EDS_last_4_digits', 'manual_eds', 'manual_patient_id', 'DOB',
                                                   'Arrival at hospital', 'Arrival time',
                                                   'Discharge date', 'Discharge time',
                                                   'Death at hospital date', 'Death at hospital time', 'Onset date', 'Onset time', 'Referral']]

In [ ]:
high_frequency_data_patient_selection_with_details.rename(columns={'Onset date': 'Stroke onset date', 'Onset time': 'Stroke onset time'}, inplace=True)

In [ ]:
high_frequency_data_patient_selection_with_details.head()

In [ ]:
save_data = False

In [ ]:
if save_data:
    high_frequency_data_patient_selection_with_details.to_csv(os.path.join(output_path, 'high_frequency_data_patient_selection_with_details.csv'))
    high_frequency_data_patient_selection_without_details = stroke_df[['patient_id', 'EDS_last_4_digits', 'DOB',
                                                   'Arrival at hospital', 'Arrival time',
                                                   'Discharge date', 'Discharge time',
                                                   'Death at hospital date', 'Death at hospital time']]
    high_frequency_data_patient_selection_without_details.to_csv(os.path.join(output_path, 'high_frequency_data_patient_selection.csv'))